# RescueLink AI - Training & Evaluation Notebook
This notebook trains and evaluates the emergency classification model, visualizes accuracy, and saves artifacts.
This notebook will guide you step by step:
1. Load the dataset
2. Tokenize the text
3. Train the model
4. Evaluate accuracy
5. Visualize results



In [1]:
import os
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizerFast

from models.emergency_classifier import EmergencyClassifier
from utils.encoders import load_encoders


C:\Users\Aaron\GitHub Repos\RescueLink\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load the Dataset
Run the next cell to load the emergency dataset from `data/emergency_dataset.csv`.
We’ll split it into training and validation sets.


In [2]:
df = pd.read_csv("data/emergency_dataset.csv")

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))


Train size: 1600
Validation size: 400


## Step 2: Tokenization
Run this cell to convert text into numerical tokens using DistilBERT’s tokenizer.
This prepares the data for the model.


In [3]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_df["text"].tolist(), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_df["text"].tolist(), truncation=True, padding=True, max_length=128)


In [4]:
from training.train import EmergencyDataset

train_dataset = EmergencyDataset(train_encodings, train_df["type_label"].tolist(), train_df["severity_label"].tolist())
val_dataset = EmergencyDataset(val_encodings, val_df["type_label"].tolist(), val_df["severity_label"].tolist())

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)


KeyError: 'type_label'

## Step 3: Training Loop
Run this cell to train the EmergencyClassifier model.
You’ll see training and validation loss after each epoch.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmergencyClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

train_losses, val_losses = [], []

for epoch in range(3):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask)
        loss_type = torch.nn.functional.cross_entropy(outputs["type_logits"], labels[:,0])
        loss_severity = torch.nn.functional.cross_entropy(outputs["severity_logits"], labels[:,1])
        loss = loss_type + loss_severity

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_losses.append(total_loss)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            loss_type = torch.nn.functional.cross_entropy(outputs["type_logits"], labels[:,0])
            loss_severity = torch.nn.functional.cross_entropy(outputs["severity_logits"], labels[:,1])
            val_loss += (loss_type + loss_severity).item()
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1} | Train Loss: {total_loss:.4f} | Val Loss: {val_loss:.4f}")


## Step 4: Accuracy & Metrics
Run this cell to evaluate the model on the validation set.
You’ll see precision, recall, and F1 scores.


In [ ]:
y_true, y_pred = [], []

model.eval()
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()
        outputs = model(input_ids, attention_mask)

        preds_type = torch.argmax(outputs["type_logits"], dim=1).cpu().numpy()
        preds_severity = torch.argmax(outputs["severity_logits"], dim=1).cpu().numpy()

        y_true.extend(labels)
        y_pred.extend(list(zip(preds_type, preds_severity)))

print(classification_report([t[0] for t in y_true], [p[0] for p in y_pred]))
print(classification_report([t[1] for t in y_true], [p[1] for p in y_pred]))


## Step 5: Confusion Matrices & Loss Curves
Run these cells to visualize:
- Where the model misclassifies (confusion matrices)
- How training progressed (loss curves)


In [ ]:
cm_type = confusion_matrix([t[0] for t in y_true], [p[0] for p in y_pred])
sns.heatmap(cm_type, annot=True, fmt="d", cmap="Blues")
plt.title("Incident Type Confusion Matrix")
plt.show()

cm_severity = confusion_matrix([t[1] for t in y_true], [p[1] for p in y_pred])
sns.heatmap(cm_severity, annot=True, fmt="d", cmap="Reds")
plt.title("Severity Confusion Matrix")
plt.show()


In [ ]:
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.legend()
plt.title("Training vs Validation Loss")
plt.show()
